# TAAC2026 - 腾讯广告算法大赛 Baseline v3

使用 HSTU 代理模型 + 时间特征

**数据**: Kaggle Dataset `galegale05/taac2026-sample-data`

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import math
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from torch.cuda.amp import autocast, GradScaler
from datetime import datetime
from tqdm import tqdm
import os

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 配置
class Config:
    data_path = '/kaggle/input/taac2026-sample-data/sample_data.parquet'
    d_model = 512
    n_heads = 64
    n_layers = 32
    d_ff = 512
    dropout = 0.01
    max_seq_len = 101
    batch_size = 32
    epochs = 10
    lr = 2e-4
    weight_decay = 1e-5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seed = 42
    test_size = 0.2
    session_threshold = 1800
    n_time_buckets = 256
    use_amp = True
    save_path = './checkpoints'

config = Config()
os.makedirs(config.save_path, exist_ok=True)
torch.manual_seed(config.seed)
np.random.seed(config.seed)
random.seed(config.seed)

In [ ]:
# 数据预处理
def build_vocab(df):
    int_counter = defaultdict(set)
    float_array_lens = {}
    max_item_id = 0
    for idx, row in df.iterrows():
        if row['item_id'] > max_item_id:
            max_item_id = row['item_id']
        for feat in row['user_feature']:
            fid = feat['feature_id']
            vtype = feat['feature_value_type']
            if 'int_value' in vtype and feat.get('int_value') is not None:
                int_counter[fid].add(feat['int_value'])
            if 'int_array' in vtype and feat.get('int_array') is not None:
                for val in feat['int_array']:
                    int_counter[fid].add(val)
            if 'float_array' in vtype and feat.get('float_array') is not None:
                arr = feat['float_array']
                if fid not in float_array_lens:
                    float_array_lens[fid] = len(arr)
        for feat in row['item_feature']:
            fid = feat['feature_id']
            vtype = feat['feature_value_type']
            if 'int_value' in vtype and feat.get('int_value') is not None:
                int_counter[fid].add(feat['int_value'])
            if 'int_array' in vtype and feat.get('int_array') is not None:
                for val in feat['int_array']:
                    int_counter[fid].add(val)
            if 'float_array' in vtype and feat.get('float_array') is not None:
                arr = feat['float_array']
                if fid not in float_array_lens:
                    float_array_lens[fid] = len(arr)
        seq_data = row['seq_feature']
        for seq_name in ['action_seq', 'content_seq', 'item_seq']:
            if seq_name not in seq_data:
                continue
            for feat in seq_data[seq_name]:
                fid = feat['feature_id']
                vtype = feat['feature_value_type']
                if 'int_array' in vtype and feat.get('int_array') is not None:
                    for val in feat['int_array']:
                        int_counter[fid].add(val)
    int_vocab = {}
    for fid, vals in int_counter.items():
        sorted_vals = sorted(vals)
        idx_map = {v: i+1 for i, v in enumerate(sorted_vals)}
        int_vocab[fid] = idx_map
    return int_vocab, float_array_lens, max_item_id

def sessionize_sequence(seq_items, seq_timestamps, threshold=1800):
    if not seq_items:
        return []
    session_ids = [0]
    current = 0
    for i in range(1, len(seq_items)):
        if seq_timestamps[i] - seq_timestamps[i-1] > threshold:
            current += 1
        session_ids.append(current)
    return session_ids

def time_difference_bucket(timestamps, max_buckets=256):
    if len(timestamps) <= 1:
        return [0] * len(timestamps)
    diffs = []
    for i in range(1, len(timestamps)):
        diff = timestamps[i] - timestamps[i-1]
        if diff <= 0:
            bucket = 0
        else:
            bucket = min(int(math.log2(diff + 1)), max_buckets - 1)
        diffs.append(bucket)
    diffs.insert(0, 0)
    return diffs

def extract_time_features(timestamp):
    dt = datetime.fromtimestamp(timestamp)
    return {
        'hour': dt.hour,
        'weekday': dt.weekday(),
        'day_of_month': dt.day,
        'day_of_year': dt.timetuple().tm_yday,
        'quarter': (dt.month - 1) // 3 + 1,
        'is_weekend': 1 if dt.weekday() >= 5 else 0
    }

In [ ]:
# 数据集
class TAACDataset(Dataset):
    def __init__(self, df, int_vocab, float_array_lens, config, is_train=True):
        self.df = df.reset_index(drop=True)
        self.int_vocab = int_vocab
        self.float_array_lens = float_array_lens
        self.config = config
        self.is_train = is_train
        self.labels = []
        self.user_features = []
        self.item_features = []
        self.seq_features = []
        self.target_item_ids = []
        self.context_time_features = []
        for idx, row in df.iterrows():
            label = 1 if row['label'][0]['action_type'] == 2 else 0
            self.labels.append(label)
            self.user_features.append(row['user_feature'])
            self.item_features.append(row['item_feature'])
            self.seq_features.append(row['seq_feature'])
            self.target_item_ids.append(row['item_id'])
            self.context_time_features.append(extract_time_features(row['timestamp']))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        label = self.labels[idx]
        user_feat = self.user_features[idx]
        item_feat = self.item_features[idx]
        seq_feat = self.seq_features[idx]
        target_item_id = self.target_item_ids[idx]
        context_time = self.context_time_features[idx]

        item_seq, action_seq, timestamp_seq = [], [], []
        if 'item_seq' in seq_feat:
            for feat in seq_feat['item_seq']:
                if feat['feature_id'] == 19 and feat.get('int_array') is not None:
                    item_seq = feat['int_array']
        if 'action_seq' in seq_feat:
            for feat in seq_feat['action_seq']:
                if feat['feature_id'] == 40 and feat.get('int_array') is not None:
                    action_seq = feat['int_array']
                if feat['feature_id'] == 41 and feat.get('int_array') is not None:
                    timestamp_seq = feat['int_array']

        min_len = min(len(item_seq), len(action_seq), len(timestamp_seq))
        if min_len > 0:
            item_seq, action_seq, timestamp_seq = item_seq[:min_len], action_seq[:min_len], timestamp_seq[:min_len]

        if len(item_seq) > self.config.max_seq_len:
            item_seq = item_seq[-self.config.max_seq_len:]
            action_seq = action_seq[-self.config.max_seq_len:]
            timestamp_seq = timestamp_seq[-self.config.max_seq_len:]

        session_ids = sessionize_sequence(item_seq, timestamp_seq, self.config.session_threshold)
        time_diffs = time_difference_bucket(timestamp_seq, self.config.n_time_buckets)

        user_int_feats, user_float_lens = {}, {}
        for feat in user_feat:
            fid = feat['feature_id']
            vtype = feat['feature_value_type']
            if 'int_value' in vtype and feat.get('int_value') is not None:
                user_int_feats[fid] = self.int_vocab.get(fid, {}).get(feat['int_value'], 0)
            if 'int_array' in vtype and feat.get('int_array') is not None:
                user_int_feats[fid] = [self.int_vocab.get(fid, {}).get(v, 0) for v in feat['int_array'][:20]]
            if 'float_array' in vtype and feat.get('float_array') is not None:
                user_float_lens[fid] = len(feat['float_array'])

        item_int_feats, item_float_lens = {}, {}
        for feat in item_feat:
            fid = feat['feature_id']
            vtype = feat['feature_value_type']
            if 'int_value' in vtype and feat.get('int_value') is not None:
                item_int_feats[fid] = self.int_vocab.get(fid, {}).get(feat['int_value'], 0)
            if 'int_array' in vtype and feat.get('int_array') is not None:
                item_int_feats[fid] = [self.int_vocab.get(fid, {}).get(v, 0) for v in feat['int_array'][:20]]
            if 'float_array' in vtype and feat.get('float_array') is not None:
                item_float_lens[fid] = len(feat['float_array'])

        seq_item_ids = [self.int_vocab.get(19, {}).get(v, 0) for v in item_seq]
        seq_action_ids = [self.int_vocab.get(40, {}).get(v, 0) for v in action_seq]
        target_item_id_mapped = self.int_vocab.get(19, {}).get(target_item_id, 0)

        return {
            'label': label,
            'user_int_feats': user_int_feats,
            'user_float_lens': user_float_lens,
            'item_int_feats': item_int_feats,
            'item_float_lens': item_float_lens,
            'seq_item_ids': seq_item_ids,
            'seq_action_ids': seq_action_ids,
            'seq_session_ids': session_ids,
            'seq_time_diffs': time_diffs,
            'target_item_id': target_item_id_mapped,
            'context_time': context_time
        }

def collate_fn(batch):
    labels = torch.tensor([x['label'] for x in batch], dtype=torch.float32)

    max_user_int_keys = max((max(x['user_int_feats'].keys(), default=0) for x in batch), default=0)
    max_item_int_keys = max((max(x['item_int_feats'].keys(), default=0) for x in batch), default=0)
    user_int_keys = list(range(1, max_user_int_keys + 1))
    item_int_keys = list(range(1, max_item_int_keys + 1))
    max_seq_len = max(len(x['seq_item_ids']) for x in batch) or 1

    user_int_feats = {kid: [x['user_int_feats'].get(kid, 0) for x in batch] for kid in user_int_keys}
    item_int_feats = {kid: [x['item_int_feats'].get(kid, 0) for x in batch] for kid in item_int_keys}
    user_float_lens = {k: [x['user_float_lens'].get(k, 0) for x in batch] for k in batch[0]['user_float_lens']}
    item_float_lens = {k: [x['item_float_lens'].get(k, 0) for x in batch] for k in batch[0]['item_float_lens']}

    user_int_tensors = {k: torch.tensor(v, dtype=torch.long) for k, v in user_int_feats.items()}
    item_int_tensors = {k: torch.tensor(v, dtype=torch.long) for k, v in item_int_feats.items()}
    user_float_tensors = {k: torch.tensor(v, dtype=torch.long) for k, v in user_float_lens.items()}
    item_float_tensors = {k: torch.tensor(v, dtype=torch.long) for k, v in item_float_lens.items()}

    seq_item_ids = torch.zeros(len(batch), max_seq_len, dtype=torch.long)
    seq_action_ids = torch.zeros(len(batch), max_seq_len, dtype=torch.long)
    seq_session_ids = torch.zeros(len(batch), max_seq_len, dtype=torch.long)
    seq_time_diffs = torch.zeros(len(batch), max_seq_len, dtype=torch.long)
    seq_mask = torch.zeros(len(batch), max_seq_len, dtype=torch.bool)

    context_hours, context_weekdays, context_day_of_months = [], [], []
    context_day_of_years, context_quarters, context_is_weekends = [], [], []

    for i, x in enumerate(batch):
        seq_len = len(x['seq_item_ids'])
        if seq_len > 0:
            seq_item_ids[i, :seq_len] = torch.tensor(x['seq_item_ids'], dtype=torch.long)
            seq_action_ids[i, :seq_len] = torch.tensor(x['seq_action_ids'], dtype=torch.long)
            seq_session_ids[i, :seq_len] = torch.tensor(x['seq_session_ids'][:seq_len], dtype=torch.long)
            seq_time_diffs[i, :seq_len] = torch.tensor(x['seq_time_diffs'][:seq_len], dtype=torch.long)
        seq_mask[i, :seq_len] = True
        ctx = x['context_time']
        context_hours.append(ctx['hour'])
        context_weekdays.append(ctx['weekday'])
        context_day_of_months.append(ctx['day_of_month'])
        context_day_of_years.append(ctx['day_of_year'])
        context_quarters.append(ctx['quarter'])
        context_is_weekends.append(ctx['is_weekend'])

    target_item_ids = torch.tensor([x['target_item_id'] for x in batch], dtype=torch.long)
    context_time_features = {
        'hour': torch.tensor(context_hours, dtype=torch.long),
        'weekday': torch.tensor(context_weekdays, dtype=torch.long),
        'day_of_month': torch.tensor(context_day_of_months, dtype=torch.long),
        'day_of_year': torch.tensor(context_day_of_years, dtype=torch.long),
        'quarter': torch.tensor(context_quarters, dtype=torch.long),
        'is_weekend': torch.tensor(context_is_weekends, dtype=torch.long)
    }

    return {
        'labels': labels,
        'user_int_feats': user_int_tensors,
        'user_float_lens': user_float_tensors,
        'item_int_feats': item_int_tensors,
        'item_float_lens': item_float_tensors,
        'seq_item_ids': seq_item_ids,
        'seq_action_ids': seq_action_ids,
        'seq_session_ids': seq_session_ids,
        'seq_time_diffs': seq_time_diffs,
        'seq_mask': seq_mask,
        'target_item_ids': target_item_ids,
        'context_time': context_time_features
    }

In [ ]:
# HSTU 模型
class HSTUModel(nn.Module):
    def __init__(self, config, num_user_int_feats, num_item_int_feats, 
                 num_user_float_feats, num_item_float_feats, max_item_id):
        super().__init__()
        self.config = config
        
        self.item_embedding = nn.Embedding(max_item_id + 2, config.d_model, padding_idx=0)
        self.action_embedding = nn.Embedding(256, config.d_model)
        self.session_embedding = nn.Embedding(256, config.d_model)
        self.time_diff_embedding = nn.Embedding(config.n_time_buckets, config.d_model)
        
        self.user_int_embeddings = nn.ModuleDict()
        self.item_int_embeddings = nn.ModuleDict()
        for fid in num_user_int_feats:
            n_cats = len(num_user_int_feats[fid]) + 1
            self.user_int_embeddings[str(fid)] = nn.Embedding(n_cats, config.d_model)
        for fid in num_item_int_feats:
            n_cats = len(num_item_int_feats[fid]) + 1
            self.item_int_embeddings[str(fid)] = nn.Embedding(n_cats, config.d_model)
        
        self.user_float_projections = nn.ModuleDict()
        self.item_float_projections = nn.ModuleDict()
        for fid, arr_len in num_user_float_feats.items():
            self.user_float_projections[str(fid)] = nn.Linear(arr_len, config.d_model)
        for fid, arr_len in num_item_float_feats.items():
            self.item_float_projections[str(fid)] = nn.Linear(arr_len, config.d_model)
        
        self.hour_embedding = nn.Embedding(24, config.d_model)
        self.weekday_embedding = nn.Embedding(7, config.d_model)
        self.day_of_month_embedding = nn.Embedding(32, config.d_model)
        self.day_of_year_embedding = nn.Embedding(366, config.d_model)
        self.quarter_embedding = nn.Embedding(5, config.d_model)
        self.is_weekend_embedding = nn.Embedding(2, config.d_model)
        
        self.time_encoder = nn.Sequential(
            nn.Linear(config.d_model * 6, config.d_model),
            nn.GELU(),
            nn.Linear(config.d_model, config.d_model)
        )
        
        self.sequence_layers = nn.ModuleList([
            HSTULayer(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])
        
        self.output_proj = nn.Sequential(
            nn.Linear(config.d_model * 3, config.d_model),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model, config.d_model // 4),
            nn.GELU(),
            nn.Linear(config.d_model // 4, 1)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def encode_user_features(self, user_int_feats, user_float_lens):
        embeddings = []
        for fid, emb_module in self.user_int_embeddings.items():
            fid_int = int(fid)
            if fid_int in user_int_feats:
                embeddings.append(emb_module(user_int_feats[fid_int]))
        for fid, linear_module in self.user_float_projections.items():
            fid_int = int(fid)
            if fid_int in user_float_lens:
                embeddings.append(linear_module(user_float_lens[fid_int].float().unsqueeze(-1)))
        if embeddings:
            return torch.stack(embeddings, dim=1).mean(dim=1)
        return torch.zeros(embeddings[0].shape[0] if embeddings else 1, self.config.d_model, device=next(self.parameters()).device)
    
    def encode_item_features(self, item_int_feats, item_float_lens):
        embeddings = []
        for fid, emb_module in self.item_int_embeddings.items():
            fid_int = int(fid)
            if fid_int in item_int_feats:
                embeddings.append(emb_module(item_int_feats[fid_int]))
        for fid, linear_module in self.item_float_projections.items():
            fid_int = int(fid)
            if fid_int in item_float_lens:
                embeddings.append(linear_module(item_float_lens[fid_int].float().unsqueeze(-1)))
        if embeddings:
            return torch.stack(embeddings, dim=1).mean(dim=1)
        return torch.zeros(embeddings[0].shape[0] if embeddings else 1, self.config.d_model, device=next(self.parameters()).device)
    
    def encode_sequence(self, seq_item_ids, seq_action_ids, seq_session_ids, seq_time_diffs, seq_mask, target_item_id):
        item_emb = self.item_embedding(seq_item_ids)
        action_emb = self.action_embedding(seq_action_ids)
        session_emb = self.session_embedding(seq_session_ids)
        time_diff_emb = self.time_diff_embedding(seq_time_diffs)
        target_emb = self.item_embedding(target_item_id)
        
        seq_emb = item_emb + action_emb + session_emb + time_diff_emb
        pos_emb = self._get_positional_embeddings(seq_emb.shape[1], seq_emb.device)
        seq_emb = seq_emb + pos_emb.uns(0)
        
        for layer in self.sequence_layers:
            seq_emb = layer(seq_emb, seq_mask)
        
        mask = seq_mask.uns(-1).float()
        seq_repr = (seq_emb * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        return seq_repr, target_emb
    
    def _get_positional_embeddings(self, seq_len, device):
        position = torch.arange(seq_len, device=device).float()
        div_term = torch.exp(torch.arange(0, self.config.d_model, 2, device=device).float() * -(math.log(10000.0) / self.config.d_model))
        pe = torch.zeros(seq_len, self.config.d_model, device=device)
        pe[:, 0::2] = torch.sin(position.uns(1) * div_term)
        pe[:, 1::2] = torch.cos(position.uns(1) * div_term)
        return pe
    
    def encode_context_time(self, context_time):
        combined = torch.cat([
            self.hour_embedding(context_time['hour']),
            self.weekday_embedding(context_time['weekday']),
            self.day_of_month_embedding(context_time['day_of_month']),
            self.day_of_year_embedding(context_time['day_of_year']),
            self.quarter_embedding(context_time['quarter']),
            self.is_weekend_embedding(context_time['is_weekend'])
        ], dim=-1)
        return self.time_encoder(combined)
    
    def forward(self, batch):
        user_repr = self.encode_user_features(batch['user_int_feats'], batch['user_float_lens'])
        item_repr = self.encode_item_features(batch['item_int_feats'], batch['item_float_lens'])
        seq_repr, target_emb = self.encode_sequence(batch['seq_item_ids'], batch['seq_action_ids'],
            batch['seq_session_ids'], batch['seq_time_diffs'], batch['seq_mask'], batch['target_item_ids'])
        context_time_repr = self.encode_context_time(batch['context_time'])
        combined = torch.cat([user_repr, item_repr + target_emb, seq_repr + context_time_repr], dim=-1)
        return self.output_proj(combined).squeeze(-1)

class HSTULayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=~mask)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x

In [ ]:
# 训练函数
def train_epoch(model, dataloader, optimizer, scaler, config, epoch):
    model.train()
    total_loss, total_bce = 0, 0
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}')
    
    for batch in pbar:
        labels = batch['labels'].to(config.device)
        optimizer.zero_grad()
        
        with autocast(enabled=config.use_amp):
            logits = model(batch)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        total_bce += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss/len(dataloader), total_bce/len(dataloader)

def evaluate(model, dataloader, config):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            labels = batch['labels'].to(config.device)
            with autocast(enabled=config.use_amp):
                logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs.flatten().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    
    return roc_auc_score(all_labels, all_preds)

In [ ]:
# 加载数据
print('加载数据...')
df = pd.read_parquet(config.data_path)
print(f'数据集大小: {len(df)}')

print('构建词表...')
int_vocab, float_array_lens, max_item_id = build_vocab(df)
print(f'特征: {len(int_vocab)}, Float特征: {len(float_array_lens)}, Max item_id: {max_item_id}')

train_df, val_df = train_test_split(df, test_size=config.test_size, random_state=config.seed)
print(f'训练集: {len(train_df)}, 验证集: {len(val_df)}')

train_dataset = TAACDataset(train_df, int_vocab, float_array_lens, config)
val_dataset = TAACDataset(val_df, int_vocab, float_array_lens, config)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)
print('数据加载完成!')

In [ ]:
# 初始化模型
print('初始化模型...')
model = HSTUModel(config, int_vocab, int_vocab, float_array_lens, float_array_lens, max_item_id).to(config.device)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 训练
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scaler = GradScaler(enabled=config.use_amp)

best_auc = 0
for epoch in range(config.epochs):
    train_loss, train_bce = train_epoch(model, train_loader, optimizer, scaler, config, epoch)
    val_auc = evaluate(model, val_loader, config)
    
    print(f'Epoch {epoch+1}/{config.epochs} | Loss: {train_loss:.4f} | BCE: {train_bce:.4f} | Val AUC: {val_auc:.4f}')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), os.path.join(config.save_path, 'best_model.pt'))
        print(f'  -> 保存最佳模型! AUC: {best_auc:.4f}')

print(f'\n训练完成! 最佳 AUC: {best_auc:.4f}')